# 05 · Example of PyTorch Dataset and DataLoader

Build a segmentation data pipeline from the MSHMM catalog.  
Split original observation indices, inspect tensor shapes, and compute an example loss over valid pixels.

### 1. Load the catalog

Set `DATA_ROOT` to the folder containing `db.xlsx` and the three batch directories.  
This example uses all catalog observations, including neighborhood observations.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from IPython.display import display

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebook":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from mshmm.catalog import load_catalog
from mshmm.split import split_catalog
from mshmm.dataset import MSHMMDataset

DATA_ROOT = REPO_ROOT.parent  # Change this to your data root.
catalog = load_catalog(DATA_ROOT)
print(f"Observations: {len(catalog):,}")

Observations: 1,686


### 2. Split original row indices

Shuffle each `sensor × is_plume` group and allocate approximately 70% / 15% / 15% to training, validation, and test sets.  
Each sensor retains approximately its original plume/non-plume proportions. Integer rounding causes small deviations from the target ratios;

In [2]:
splits = split_catalog(catalog, seed=42)
split_summary = pd.DataFrame({
    "observations": {name: len(indices) for name, indices in splits.items()}
})
split_summary["percent"] = (split_summary["observations"] / len(catalog) * 100).round(2)
display(split_summary)

print("Example training indices:", splits["train"][:10].tolist())
print("Shared train/val indices:", len(set(splits["train"]) & set(splits["val"])))
print("Shared train/test indices:", len(set(splits["train"]) & set(splits["test"])))
print("Shared val/test indices:", len(set(splits["val"]) & set(splits["test"])))

,observations,percent
train,1181,70.05
val,253,15.01
test,252,14.95


Example training indices: [1350, 99, 676, 802, 496, 157, 1057, 1377, 79, 571]
Shared train/val indices: 0
Shared train/test indices: 0
Shared val/test indices: 0


### 3. Check sensor and annotation proportions

The first table counts observations for each sensor and annotation group in every split.  
The second table shows the plume percentage within each sensor, comparing the full catalog against the three subsets.

In [3]:
subsets = {name: catalog.loc[indices] for name, indices in splits.items()}
counts = pd.concat({
    name: subset.groupby(["sensor", "is_plume"]).size()
    for name, subset in subsets.items()
}, axis=1).fillna(0).astype(int)
display(counts)

plume_percent = pd.concat({
    name: subset.groupby("sensor")["is_plume"].mean() * 100
    for name, subset in {"all": catalog, **subsets}.items()
}, axis=1)
display(plume_percent.round(2).rename_axis("sensor"))

train  val  test
sensor is_plume                  
EMIT   False       398   85    86
       True         85   18    18
ENMAP  False        98   21    21
       True         38    8     8
GF5    False       335   72    71
       True        110   24    23
PRS    False        83   18    17
       True         34    7     8

,all,train,val,test
sensor,,,,
EMIT,17.54,17.60,17.48,17.31
ENMAP,27.84,27.94,27.59,27.59
GF5,24.72,24.72,25.00,24.47
PRS,29.34,29.06,28.00,32.00


### 4. Create datasets and inspect a sample

The dataset reads `RGB.npy`, `1650.npy`, `1730.npy`, `CH4.npy`, and `plume_mask.npy`.       
 `valid_mask` follows the RGB/CH4 finite-value and `-9999` rules introduced in Notebook 04.  

In [4]:
datasets = {
    name: MSHMMDataset(catalog, DATA_ROOT, indices)
    for name, indices in splits.items()
}
sample = datasets["train"][0]

sample_info = pd.DataFrame([
    {"key": name, "shape": tuple(value.shape), "dtype": str(value.dtype)}
    for name, value in sample.items() if isinstance(value, torch.Tensor)
]).set_index("key")
display(sample_info)
print("Original catalog index:", sample["index"])
print(f"Valid pixel fraction: {sample['valid_mask'].float().mean().item():.1%}")

,shape,dtype
key,,
rgb,"(3, 512, 512)",torch.float32
swir,"(2, 512, 512)",torch.float32
ch4,"(1, 512, 512)",torch.float32
mask,"(1, 512, 512)",torch.float32
valid_mask,"(1, 512, 512)",torch.bool


Original catalog index: 1350
Valid pixel fraction: 94.5%


### 5. Create DataLoaders

Use shuffling for training and fixed ordering for validation and testing.  

In [5]:
loaders = {
    name: DataLoader(
        dataset,
        batch_size=4,
        shuffle=(name == "train"),
        num_workers=0,
        generator=torch.Generator().manual_seed(42),
    )
    for name, dataset in datasets.items()
}
batch = next(iter(loaders["train"]))

display(pd.DataFrame([
    {"key": name, "shape": tuple(value.shape), "dtype": str(value.dtype)}
    for name, value in batch.items()
]).set_index("key"))
print("Batch catalog indices:", batch["index"].tolist())

,shape,dtype
key,,
rgb,"(4, 3, 512, 512)",torch.float32
swir,"(4, 2, 512, 512)",torch.float32
ch4,"(4, 1, 512, 512)",torch.float32
mask,"(4, 1, 512, 512)",torch.float32
valid_mask,"(4, 1, 512, 512)",torch.bool
index,"(4,)",torch.int64


Batch catalog indices: [712, 936, 813, 1029]


### 6. Prepare model inputs and a masked segmentation loss

Choose normalization to suit your model before training. The dataset itself does not normalize or clip valid values.  
Below, RGB, SWIR, and CH4 are concatenated into a six-channel tensor solely to show the channel arrangement.

The zero logits below are a placeholder for a model output, not a trained prediction.  
Binary cross-entropy is evaluated only at `valid_mask=True` pixels, including both annotated plume pixels and valid background pixels. Invalid inputs have already been zero-filled before model inference.

In [6]:
inputs = torch.cat([batch["rgb"], batch["swir"], batch["ch4"]], dim=1)
print("Combined input shape:", tuple(inputs.shape))

# Replace these placeholder logits with model(inputs) in a training script.
logits = torch.zeros_like(batch["ch4"], requires_grad=True)
valid = batch["valid_mask"]
loss = F.binary_cross_entropy_with_logits(logits[valid], batch["mask"][valid])
loss.backward()

print(f"Valid pixels in batch: {valid.sum().item():,}")
print(f"Example segmentation loss: {loss.item():.4f}")

Combined input shape: (4, 6, 512, 512)
Valid pixels in batch: 941,065
Example segmentation loss: 0.6931
